# ReviewSense PT-BR — 04 · Comparação final independente

Este notebook só reabre métricas agregadas já salvas. Não lê o teste, carrega modelos, treina ou repete inferência. O protocolo foi congelado antes da execução final.

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from reviewsense.final_reporting import comparison_table
from reviewsense.evaluate import plot_confusion_matrix

candidates = [Path(os.environ["REVIEWSENSE_REPO"])] if "REVIEWSENSE_REPO" in os.environ else [Path.cwd(), Path.cwd().parent]
ROOT = next(p.resolve() for p in candidates if (p / "configs/phase3c_final.json").exists())
result = json.loads((ROOT / "reports/phase3c_final_metrics.json").read_text())
assert result["status"] == "completed"
print("Commit congelado:", result["git"]["commit"])
print("Execução:", result["started_at"], "até", result["finished_at"])
print("Exemplos:", result["test"]["rows"], "Leituras do CSV:", result["test"]["read_opens"])
display(pd.DataFrame(comparison_table(result)).set_index("modelo"))
display(pd.Series(result["comparison"]))

## Classes e matrizes

Linhas reais e colunas previstas, ordem negativo/neutro/positivo. F1 Macro dá peso igual às classes. Não há seleção de modelo com os resultados do teste.

In [ ]:
display(pd.concat({name: pd.DataFrame(row["metrics"]["per_class"]).T
                   for name, row in result["models"].items()}, names=["modelo", "classe"]))
for name, row in result["models"].items():
    figure = plot_confusion_matrix(row["metrics"], title=f"{name} — teste final")
    display(figure)
    plt.close(figure)
neutral = pd.DataFrame({name: {"corretos": row["metrics"]["confusion_matrix"][1][1],
                            "para_negativo": row["metrics"]["confusion_matrix"][1][0],
                            "para_positivo": row["metrics"]["confusion_matrix"][1][2]}
                        for name, row in result["models"].items()}).T
display(neutral)
display(pd.Series(result["paired_agreement"]))

## Inferência e custo histórico

Benchmark pré-teste: 1.024 textos de validação, três repetições após warm-up. Tempos do teste pertencem à passagem única. ms/texto é amortizado em batch, não latência individual. RSS inclui o processo com ambos os modelos; CPU e GPU são infraestruturas diferentes.

In [ ]:
benchmark = result["preflight"]["benchmark"]
display(pd.DataFrame({name: {key: row[key] for key in ["batch_size", "median_seconds", "texts_per_second", "milliseconds_per_text_amortized"]}
                     for name, row in benchmark.items()}).T)
display(pd.DataFrame({name: {key: row["inference"][key] for key in ["samples", "seconds", "texts_per_second", "milliseconds_per_text_amortized", "process_rss_peak_sampled_bytes"]}
                     for name, row in result["models"].items()}).T)
display(pd.Series(result["models"]["bertimbau"]["inference"]["pytorch"]))
display(pd.Series(result["historical_training"]))

## Integridade e limites

O JSON contém hashes de modelos, código, configuração, protocolo e referências. Não contém avaliações ou IDs individuais. Estrelas são proxy; há truncamento, domínio de produtos de 2018 e possível sobreposição de autores/produtos ou quase duplicatas. A comparação é descritiva, sem comprovação automática de significância. O teste deixa de ser informação não observada após esta avaliação e não deve orientar novos ajustes no experimento.

In [ ]:
display(pd.Series(result["test"]))
display(pd.Series(result["software"]))
print("Candidato principal definido antes do teste:", result["principal_candidate_preselected"])
print("Protocolo: docs/FASE3C_PROTOCOL.md")
print("Relatório: reports/validation_phase3c.md")